In [1]:
import os
import pathlib
import datetime as dt
from typing import Dict, List

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Define output directory relative to the notebook path
DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Load environment configuration from .env if available
load_dotenv()
print("Setup complete. Raw data folder active at:", DATA_RAW.resolve())

Setup complete. Raw data folder active at: /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework04/notebooks/data/raw


In [2]:
def safe_stamp() -> str:
    """Generates standard timestamp string YYYYMMDD-HHMMSS."""
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    """Generates a structured, reproducible raw filename."""
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    """Validates dataframe schema integrity, missing columns, and data coercion."""
    report = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        report['missing_cols'] = f"Missing columns: {missing}"
        
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                report[f'dtype_{col}'] = f"Coercion failed for {col} to {dtype}: {e}"
                
    report['na_total'] = int(df.isna().sum().sum())
    report['shape'] = list(df.shape)
    return report

In [3]:
SYMBOL = "PLTR"
SOURCE = "yfinance"

print(f"Fetching 6 months of daily market data for {SYMBOL}...")

# Download data via yfinance
df_raw = yf.download(
    SYMBOL, 
    period="6mo", 
    interval="1d", 
    auto_adjust=False, 
    multi_level_index=False
).reset_index()

# Extract and standardize required columns
df_api = df_raw[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
df_api.columns = ['date', 'open', 'high', 'low', 'close', 'volume']

# Coerce data types
df_api['date'] = pd.to_datetime(df_api['date'])
for col in ['open', 'high', 'low', 'close', 'volume']:
    df_api[col] = pd.to_numeric(df_api[col], errors='coerce')

df_api = df_api.sort_values('date').reset_index(drop=True)

# Run schema validation
required_api_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
dtypes_api = {col: 'float' for col in ['open', 'high', 'low', 'close', 'volume']}
dtypes_api['date'] = 'datetime64[ns]'

api_val = validate_df(df_api, required_cols=required_api_cols, dtypes_map=dtypes_api)
print("API Validation Report:", api_val)

# Save output to data/raw/
api_fname = safe_filename(prefix="api", meta={"source": SOURCE, "symbol": SYMBOL})
api_out_path = DATA_RAW / api_fname
df_api.to_csv(api_out_path, index=False)
print("Saved API dataset to:", api_out_path)

Fetching 6 months of daily market data for PLTR...


[*********************100%***********************]  1 of 1 completed

API Validation Report: {'na_total': 0, 'shape': [125, 6]}
Saved API dataset to: data/raw/api_source-yfinance_symbol-PLTR_20260817-230149.csv


In [4]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
SITE_NAME = "wikipedia"
TABLE_NAME = "sp500"

headers = {"User-Agent": "AFE-Course-Notebook/1.0 (contact: student@nyu.edu)"}

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Locate the constituent table
    table = soup.find('table', id='constituents') or soup.find('table', class_='wikitable')
    if table is None:
        raise RuntimeError("Target table not found on page")
        
    rows = []
    for tr in table.find_all('tr'):
        cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
        if cells:
            rows.append(cells)
            
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

except Exception as e:
    print(f"Scrape encountered issue ({e}). Loading inline fallback table...")
    html_fallback = """
    <table>
      <tr><th>Symbol</th><th>Security</th><th>GICS Sector</th></tr>
      <tr><td>PLTR</td><td>Palantir Technologies Inc.</td><td>Information Technology</td></tr>
      <tr><td>AAPL</td><td>Apple Inc.</td><td>Information Technology</td></tr>
      <tr><td>MSFT</td><td>Microsoft Corp.</td><td>Information Technology</td></tr>
    </table>
    """
    soup = BeautifulSoup(html_fallback, 'html.parser')
    rows = [[td.get_text(strip=True) for td in tr.find_all(['td', 'th'])] for tr in soup.find_all('tr')]
    header, *data = rows
    df_scrape = pd.DataFrame(data, columns=header)

# Optional numeric/type cleaning
for col in df_scrape.columns:
    if col.lower() in ['cik', 'price']:
        df_scrape[col] = pd.to_numeric(df_scrape[col], errors='coerce')

# Validate scraped data
scrape_val = validate_df(df_scrape, required_cols=list(df_scrape.columns[:3]), dtypes_map={})
print("Scrape Validation Report:", scrape_val)

# Save output to data/raw/
scrape_fname = safe_filename(prefix="scrape", meta={"site": SITE_NAME, "table": TABLE_NAME})
scrape_out_path = DATA_RAW / scrape_fname
df_scrape.to_csv(scrape_out_path, index=False)
print("Saved scraped dataset to:", scrape_out_path)

Scrape Validation Report: {'na_total': 0, 'shape': [502, 8]}
Saved scraped dataset to: data/raw/scrape_site-wikipedia_table-sp500_20260817-230151.csv


### Stage 04 Homework Ingestion Summary

**Data Sources & Parameters:**
1. **API Dataset (`PLTR`):**
   - **Provider:** Yahoo Finance (`yfinance` module)
   - **Parameters:** Ticker `PLTR`, 6-month historical lookback (`period="6mo"`), daily interval (`1d`).
   - **Validation:** Ensured standard column names (`date`, `open`, `high`, `low`, `close`, `volume`), verified numeric types, and checked zero unexpected missing values.

2. **Scraped Table:**
   - **Source:** Wikipedia S&P 500 Constituent Table
   - **Extraction:** Parsed with `requests` + `BeautifulSoup` (with custom `User-Agent` headers).
   - **Validation:** Checked shape, required header fields, and total missing value count.

**Assumptions & Risks:**
- **Rate Limits & API Drift:** Relying on unauthenticated wrappers (`yfinance`) poses risks if source schema structures or endpoint definitions change.
- **Scraper Fragility:** BeautifulSoup parsing relies on DOM element IDs/classes (`id='constituents'`). Website UI design overhauls can break structural assumptions, which is handled via exception try/except fallbacks.
- **Secrets:** `.env` file management is configured via `python-dotenv` and ignored in Git settings to prevent credential exposure.